# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all entities by their `@id` as per the Croissant specification.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

    https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata (using the class attribute, not subscript)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

if hasattr(metadata, 'keywords'):
    print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset may have one or more record sets (tables). We'll list their `@id` and summary information, all by Croissant `@id`.

In [ ]:
# List available record sets and their field/column @id references
record_sets = list(dataset.record_sets)
print(f"Record sets (@id): {[rs.id for rs in record_sets]}")

for rs in record_sets:
    print(f"\nRecord Set: {rs.id}\n  Name: {rs.name}\n  Description: {rs.description}")
    print("  Available fields (by @id):")
    # Each field is identified by its Croissant @id
    for field in rs.fields:
        col_id = getattr(field, 'column', None)
        col_info = f" - column @id: {col_id.id}" if col_id else ""
        print(f"    Field @id: {field.id} | Name: {field.name}{col_info}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All selections reference record set and field `@id`s from the overview. For this example, we'll select the main protein data record set.

Replace the `main_record_set_id` variable below with the desired record set `@id` from your dataset overview.

In [ ]:
# Choose a record set by @id (from previous cell)
# For this dataset, the main quantitative table record set is often '@id': 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/table1-records'
# We'll autodetect if only one record set exists
if record_sets:
    main_record_set = record_sets[0]
    main_record_set_id = main_record_set.id
    print(f"Using main record set @id: {main_record_set_id}")
else:
    raise RuntimeError("No record sets found in the dataset.")

# Load the full record set as a list of dicts, then to a DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"DataFrame columns (@id): {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping. Use field `@id`s for all code references.

Suppose this dataset has columns for protein molecular weight (MW), peptide counts, and accession. We'll look for numeric fields by their `@id`, pick one for filtering/normalizing (for example, the peptide count), and group results by another field (for example, accession or description if present).

> **Tip:** Refer to the overview table for the exact field `@id`s to use in your code. You'll likely see something like `'https://sen.science/doi/10.71728/senscience.vq4a-28xa/field_peptide_count'`.

In [ ]:
# Identify numeric fields by @id
numeric_field_id = None
possible_numeric_fields = ['peptide_count', 'molecular_weight', 'MW', 'abundance', 'score', 'coverage']
for col in df.columns:
    for p in possible_numeric_fields:
        if p in col.lower():
            numeric_field_id = col
            break
    if numeric_field_id:
        break
if not numeric_field_id:
    print("Warning: No numeric field identified. Please set 'numeric_field_id' below using a field @id.")
# For demonstration, force an assignment if not found:
# numeric_field_id = '<your_numeric_field_id>'
print(f"Using numeric field @id: {numeric_field_id}")

# Filter records with peptide count (or chosen field) > threshold
threshold = 10
if numeric_field_id and numeric_field_id in df.columns:
    # Ignore conversion errors (e.g., if not numeric)
    vals = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[vals > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)}")

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        vals[vals > threshold] - vals[vals > threshold].mean()
    ) / (vals[vals > threshold].std())
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by another field (e.g., accession @id)
    group_field_id = None
    possible_group_fields = ['accession', 'description', 'protein', 'gene']
    for col in df.columns:
        for p in possible_group_fields:
            if p in col.lower():
                group_field_id = col
                break
        if group_field_id:
            break
    if group_field_id and group_field_id in filtered_df.columns:
        # Group by this categorical field and show group means
        grouped_df = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field identified for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll draw a histogram of the numeric field and (if available) a boxplot grouped by a chosen categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    nums = pd.to_numeric(df[numeric_field_id], errors='coerce').dropna()
    plt.figure(figsize=(8,4))
    sns.histplot(nums, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    # Boxplot grouped by a categorical field (if found)
    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        top_categories = df[group_field_id].value_counts().index[:5]  # Show top 5 groups
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id][df[group_field_id].isin(top_categories)],
                    y=nums[df[group_field_id].isin(top_categories)])
        plt.title(f"{numeric_field_id} by {group_field_id} (top 5 groups)")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to programmatically load the >[FAIR^2]< Croissant dataset for "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells". We walked through metadata inspection, record set field discovery (by `@id`), and exploratory data analysis including filtering, normalization, and grouping. We visualized important numeric distributions and can now proceed with more advanced analysis or modeling as required. All data elements were referenced by their unique `@id`, ensuring full traceability to the linked schema components.